In [ ]:
!pip install opencv-python keras numpy matplotlib tqdm tensorflow

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
import tensorflow as tf

In [ ]:
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu,True)

In [ ]:
POS_PATH = os.path.join('data' , 'positive')
ANC_PATH = os.path.join('data' , 'anchor')
NEG_PATH = os.path.join('data' , 'negative')

In [ ]:
if not os.path.exists(POS_PATH):
    os.makedirs(POS_PATH)
if not os.path.exists(ANC_PATH):
    os.makedirs(ANC_PATH)
if not os.path.exists(NEG_PATH):
    os.makedirs(NEG_PATH)

In [ ]:
!tar -xf lfw.tgz

In [ ]:
# Move LFW Images to the following repository data/negative
for directory in os.listdir('lfw'):
    for file in os.listdir(os.path.join('lfw', directory)):
        EX_PATH = os.path.join('lfw', directory, file)
        NEW_PATH = os.path.join(NEG_PATH, file)
        os.replace(EX_PATH, NEW_PATH)

# Start from here

In [ ]:
anchor = tf.data.Dataset.list_files(ANC_PATH + '\\*.jpg')
positive = tf.data.Dataset.list_files(POS_PATH + '\\*.jpg')
negative = tf.data.Dataset.list_files(NEG_PATH + '\\*.jpg')

In [ ]:
def preprocess(file_path):

    # Read in image from file path
    byte_img = tf.io.read_file(file_path)
    # Load in the image
    img = tf.io.decode_jpeg(byte_img)

    # Preprocessing steps - resizing the image to be 100x100x3
    img = tf.image.resize(img, (100,100))
    # Scale image to be between 0 and 1
    img = img / 255.0

    # Return image
    return img

In [ ]:
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))))
data = positives.concatenate(negatives)

In [ ]:
def preprocess_twin(input_img, validation_img, label):
    return(preprocess(input_img), preprocess(validation_img), label)

In [ ]:
# Build dataloader pipeline
data = data.map(preprocess_twin)
data = data.cache()
data = data.shuffle(buffer_size=10000)

In [ ]:
# Training partition
train_data = data.take(round(len(data)*.7))
train_data = train_data.batch(16)
train_data = train_data.prefetch(8)

In [ ]:
# Testing partition
test_data = data.skip(round(len(data)*.7))
test_data = test_data.take(round(len(data)*.3))
test_data = test_data.batch(16)
test_data = test_data.prefetch(8)

In [ ]:
from typing import Tuple
from keras._tf_keras.keras.layers import (
    Input,
    Conv2D,
    BatchNormalization,
    Activation,
    MaxPooling2D,
    ZeroPadding2D,
    Add,
    AveragePooling2D,
    Flatten,
    Dense,
)
from keras._tf_keras.keras.initializers import glorot_uniform
from keras._tf_keras.keras.models import Model



def identity_block(X, level, block, filters):
    """
    Implementation of the identity block as defined in the ResNet model.

    Arguments:
    X -- input tensor of shape (m, n_H_prev, n_W_prev, n_C_prev)
    level -- level/layer of the block (used for naming)
    block -- block identifier (used for naming)
    filters -- python list of integers, defining the number of filters in the CONV layers

    Returns:
    X -- output of the identity block, tensor of shape (m, n_H, n_W, n_C)
    """

    # Retrieve filters
    f1, f2, f3 = filters

    # Define names for the layers using the provided level and block numbers
    conv_name = 'res' + str(level) + '_' + str(block) + '_{}_{}'

    # Save the input value to be used in the skip connection
    X_shortcut = X

    # First component of main path
    X = Conv2D(filters=f1, kernel_size=(1, 1), strides=(1, 1), padding='valid',
               name=conv_name.format(1, 'conv'), kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=conv_name.format(1, 'bn'))(X)
    X = Activation('relu')(X)

    # Second component of main path
    X = Conv2D(filters=f2, kernel_size=(3, 3), strides=(1, 1), padding='same',
               name=conv_name.format(2, 'conv'), kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=conv_name.format(2, 'bn'))(X)
    X = Activation('relu')(X)

    # Third component of main path
    X = Conv2D(filters=f3, kernel_size=(1, 1), strides=(1, 1), padding='valid',
               name=conv_name.format(3, 'conv'), kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=conv_name.format(3, 'bn'))(X)

    # Add shortcut value to main path and pass it through a ReLU activation
    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)

    return X


def convolutional_block(X, level, block, filters, s=2):
    """
    Implementation of the convolutional block as defined in the ResNet model.

    Arguments:
    X -- input tensor of shape (m, n_H_prev, n_W_prev, n_C_prev)
    level -- level/layer of the block (used for naming)
    block -- block identifier (used for naming)
    filters -- python list of integers, defining the number of filters in the CONV layers
    s -- stride for the first convolutional layer in the block

    Returns:
    X -- output of the convolutional block, tensor of shape (m, n_H, n_W, n_C)
    """

    # Retrieve filters
    f1, f2, f3 = filters

    # Define names for the layers using the provided level and block numbers
    conv_name = 'res' + str(level) + '_' + str(block) + '_{}_{}'

    # Save the input value to be used in the skip connection
    X_shortcut = X

    # First component of main path
    X = Conv2D(filters=f1, kernel_size=(1, 1), strides=(s, s), padding='valid',
               name=conv_name.format(1, 'conv'), kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=conv_name.format(1, 'bn'))(X)
    X = Activation('relu')(X)

    # Second component of main path
    X = Conv2D(filters=f2, kernel_size=(3, 3), strides=(1, 1), padding='same',
               name=conv_name.format(2, 'conv'), kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=conv_name.format(2, 'bn'))(X)
    X = Activation('relu')(X)

    # Third component of main path
    X = Conv2D(filters=f3, kernel_size=(1, 1), strides=(1, 1), padding='valid',
               name=conv_name.format(3, 'conv'), kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis=3, name=conv_name.format(3, 'bn'))(X)

    # Shortcut path
    X_shortcut = Conv2D(filters=f3, kernel_size=(1, 1), strides=(s, s), padding='valid',
                        name=conv_name.format('shortcut', 'conv'), kernel_initializer=glorot_uniform(seed=0))(X_shortcut)
    X_shortcut = BatchNormalization(axis=3, name=conv_name.format('shortcut', 'bn'))(X_shortcut)

    # Add shortcut value to main path and pass it through a ReLU activation
    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)

    return X


def ResNet50(input_size: Tuple[int, int, int] = [100,100,3], embedding_size: int = 512) -> Model:
    """
    Implementation of the ResNet50 architecture.

    Arguments:
    input_size -- shape of the input images (height, width, channels)
    embedding_size -- size of the embedding vector

    Returns:
    model -- a Model() instance in Keras
    """

    # Define the input placeholder as a tensor with shape input_size
    X_input = Input(input_size)

    # Zero-padding
    X = ZeroPadding2D((3, 3))(X_input)

    # Stage 1
    X = Conv2D(64, (7, 7), strides=(2, 2), name='conv1')(X)
    X = BatchNormalization(axis=3, name='bn_conv1')(X)
    X = Activation('relu')(X)
    X = MaxPooling2D((3, 3), strides=(2, 2))(X)

    # Stage 2
    X = convolutional_block(X, level=2, block=1, filters=[64, 64, 256], s=1)
    X = identity_block(X, level=2, block=2, filters=[64, 64, 256])
    X = identity_block(X, level=2, block=3, filters=[64, 64, 256])

    # Stage 3
    X = convolutional_block(X, level=3, block=1, filters=[128, 128, 512], s=2)
    X = identity_block(X, level=3, block=2, filters=[128, 128, 512])
    X = identity_block(X, level=3, block=3, filters=[128, 128, 512])
    X = identity_block(X, level=3, block=4, filters=[128, 128, 512])

    # Stage 4
    X = convolutional_block(X, level=4, block=1, filters=[256, 256, 1024], s=2)
    X = identity_block(X, level=4, block=2, filters=[256, 256, 1024])
    X = identity_block(X, level=4, block=3, filters=[256, 256, 1024])
    X = identity_block(X, level=4, block=4, filters=[256, 256, 1024])
    X = identity_block(X, level=4, block=5, filters=[256, 256, 1024])
    X = identity_block(X, level=4, block=6, filters=[256, 256, 1024])

    # Stage 5
    X = convolutional_block(X, level=5, block=1, filters=[512, 512, 2048], s=2)
    X = identity_block(X, level=5, block=2, filters=[512, 512, 2048])
    X = identity_block(X, level=5, block=3, filters=[512, 512, 2048])

    # Average pooling
    X = AveragePooling2D((2, 2), name='avg_pool')(X)

    # Flatten the output
    X = Flatten()(X)

    # Fully connected layer
    X = Dense(embedding_size, activation='linear', name='fc' + str(embedding_size),
              kernel_initializer=glorot_uniform(seed=0))(X)

    # Create model instance
    model = Model(inputs=X_input, outputs=X, name='ResNet50')

    return model

In [ ]:
from keras._tf_keras.keras.layers import Layer

# Siamese L1 Distance class
class L1Dist(Layer):

    # Init method - inheritance
    def __init__(self, **kwargs):
        super().__init__()

    # Magic happens here - similarity calculation
    def call(self, input_embedding, validation_embedding):
        return tf.math.abs(input_embedding - validation_embedding)

In [ ]:
embedding = ResNet50()
def make_siamese_model():
    # Anchor image input in the network
    input_image = Input(name='input_img', shape=(100, 100, 3))

    # Validation image in the network
    validation_image = Input(name='validation_img', shape=(100, 100, 3))

    # Get embeddings using your ResNet model
    inp_embedding = embedding(input_image)
    val_embedding = embedding(validation_image)

    l1 = L1Dist()
    distances = l1(inp_embedding, val_embedding)

    # Classification layer
    classifier = Dense(1, activation='sigmoid')(distances)

    return Model(inputs=[input_image, validation_image], outputs=classifier, name='SiameseNetwork')

In [ ]:
siamese_network = make_siamese_model()

In [ ]:
siamese_network.summary()

Model: "SiameseNetwork"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_img           │ (None, 100, 100,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ validation_img      │ (None, 100, 100,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ResNet50            │ (None, 512)       │ 24,636,800 │ input_img[0][0],  │
│ (Functional)        │                   │            │ validation_img[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ l1_dist (L1Dist)    │ (None, 512)       │          0 │ ResNet50[0][0],   │
│                     │                   │            │ ResNet50[1][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │        513 │ l1_dist[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,637,313 (93.98 MB)

 Trainable params: 24,584,193 (93.78 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [ ]:
from keras._tf_keras.keras.losses import BinaryCrossentropy
from keras._tf_keras.keras.optimizers import Adam

In [ ]:
binary_cross_loss = BinaryCrossentropy()

opt = Adam(1e-4)

In [ ]:
checkpoint_dir = '.\\training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt')
checkpoint = tf.train.Checkpoint(opt=opt, siamese_model=siamese_network)

In [ ]:
@tf.function
def train_step(batch):

    # Record all of our operations
    with tf.GradientTape() as tape:
        # Get anchor and positive/negative image
        X = batch[:2]
        # Get label
        y = batch[2]

        # Forward pass
        yhat = siamese_network(X, training=True)
        # Calculate loss
        loss = binary_cross_loss(y, yhat)
    print(loss)

    # Calculate gradients
    grad = tape.gradient(loss, siamese_network.trainable_variables)

    # Calculate updated weights and apply to siamese model
    opt.apply_gradients(zip(grad, siamese_network.trainable_variables))

    # Return loss
    return loss